# Dimensionality Reduction & Clustering Sensitivity Analysis

Addresses reviewer comment: *"Have the authors evaluated alternative dimensionality reduction methods or clustering methods?"*

## Part 1: Dimensionality Reduction Sensitivity
Compare PCA vs no-PCA (scale only) as input to Harmony, keeping clustering fixed (Leiden).

## Part 2: Clustering Method Sensitivity
Compare Leiden, Louvain, K-Means, Hierarchical (Ward), and Spectral clustering on the
same Harmony-corrected embedding.

## Evaluation
Pairwise ARI and NMI across all methods, plus a 3-column composite figure
(UMAP / spatial / RNA z-score heatmap) following the layout used in
`Random_Clustering_reproducibility.ipynb`.

In [ ]:
import anndata as ad
import os
import numpy as np
import scanpy as sc
import squidpy as sq
import harmonypy as hm
import umap
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist
from matplotlib.lines import Line2D

## 1. Load and preprocess data

In [ ]:
# Load nuclear protein intensity matrices of the two brain sections.
# Brain 1 (~178927 cells, 18 markers) and Brain 2 (~194890 cells, 15 markers)
# are pre-saved h5ad snapshots produced by the upstream segmentation pipeline.
adata_1st_brain = ad.read_h5ad('input/adata_18_nuclear_46_prot_brain1.h5ad')
adata_2nd_brain = ad.read_h5ad('input/brain2_nuclear_int_18prot.h5ad')

# Prefix obs_names so we can disambiguate same-numbered cells across brains
# AFTER concat (this prevents the row-order alignment bugs we hit earlier).
adata_1st_brain.obs_names = 'b1_' + adata_1st_brain.obs_names.astype(str).str.strip()
adata_2nd_brain.obs_names = 'b2_' + adata_2nd_brain.obs_names.astype(str).str.strip()

common_vars = adata_1st_brain.var_names.intersection(adata_2nd_brain.var_names)
adata_1 = adata_1st_brain[:, common_vars].copy()
adata_2 = adata_2nd_brain[:, common_vars].copy()

In [ ]:
# Load RNA matrix for the analysis brain
rna_df = pd.read_csv("input/cell_by_transcript_gene_name_matrix2.csv", index_col=0)
rna_df.index = 'b2_' + rna_df.index.astype(str).str.strip()

In [ ]:
# Combine
adata_both = ad.concat(
    [adata_1, adata_2], join="outer", label="dataset",
    keys=["1st_brain", "2nd_brain"],
)
adata_both.obs_names_make_unique()   # belt-and-suspenders -- prefixes should already disambiguate

adata_both.layers["counts"] = adata_both.X.copy()
sc.pp.normalize_total(adata_both, inplace=True)
sc.pp.log1p(adata_both)

print(adata_both)
print(f"Features: {adata_both.var_names.tolist()}")
print(f"obs_names unique: {adata_both.obs_names.is_unique}")

---
# Part 1: Dimensionality Reduction Sensitivity
## Pipeline A: PCA -> Harmony (original pipeline)

In [ ]:
# Pipeline A: PCA -> Harmony
adata_pca = adata_both.copy()
sc.tl.pca(adata_pca, svd_solver="arpack")

# takes ~ 30 mins for our dataset
ho_pca = hm.run_harmony(
    adata_pca.obsm['X_pca'], adata_pca.obs, 'dataset',
    theta=6, lamb=0.3, sigma=0.02, nclust=100,
    max_iter_harmony=20, random_state=42,
)
adata_pca.obsm['X_harmony'] = ho_pca.Z_corr.T
print(f"Pipeline A (PCA): Harmony embedding shape = {adata_pca.obsm['X_harmony'].shape}")

In [ ]:
# Pipeline A: UMAP, neighbors, Leiden
X_pca_1 = adata_pca[adata_pca.obs['dataset'] == '1st_brain'].obsm['X_harmony']
X_pca_2 = adata_pca[adata_pca.obs['dataset'] == '2nd_brain'].obsm['X_harmony']

reducer_a = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
reducer_a.fit(X_pca_2)
X_umap_a = np.vstack([reducer_a.transform(X_pca_1), reducer_a.transform(X_pca_2)])
adata_pca.obsm['X_umap'] = X_umap_a

sc.pp.neighbors(adata_pca, n_neighbors=10, random_state=42, use_rep='X_harmony')
sc.tl.leiden(adata_pca, resolution=0.8, random_state=38)

print(f"Pipeline A clusters: {adata_pca.obs['leiden'].nunique()}")

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_pca, color=["dataset"], size=5, alpha=0.9)

In [ ]:
# Save / reload (commented; uncomment to skip the heavy Harmony rerun)
adata_pca.write('output/adata_plot_harmony_pca_2_brains.h5ad')
# adata_pca = ad.read('output/adata_plot_harmony_pca_2_brains.h5ad')

## Pipeline B: Scale (no PCA) -> Harmony

In [ ]:
# Pipeline B: Scale -> Harmony (no PCA)
adata_nopca = adata_both.copy()
adata_nopca.layers["log1p"] = adata_nopca.X.copy()
sc.pp.scale(adata_nopca)

ho_nopca = hm.run_harmony(
    adata_nopca.X, adata_nopca.obs, 'dataset',
    theta=6, lamb=0.3, sigma=0.02, nclust=100,
    max_iter_harmony=20, random_state=42,
)
adata_nopca.obsm['X_harmony'] = ho_nopca.Z_corr.T
print(f"Pipeline B (no PCA): Harmony embedding shape = {adata_nopca.obsm['X_harmony'].shape}")

In [ ]:
# Pipeline B: UMAP, neighbors, Leiden
X_nopca_1 = adata_nopca[adata_nopca.obs['dataset'] == '1st_brain'].obsm['X_harmony']
X_nopca_2 = adata_nopca[adata_nopca.obs['dataset'] == '2nd_brain'].obsm['X_harmony']

reducer_b = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
reducer_b.fit(X_nopca_2)
X_umap_b = np.vstack([reducer_b.transform(X_nopca_1), reducer_b.transform(X_nopca_2)])
adata_nopca.obsm['X_umap'] = X_umap_b

sc.pp.neighbors(adata_nopca, n_neighbors=10, random_state=42, use_rep='X_harmony')
sc.tl.leiden(adata_nopca, resolution=0.8, random_state=38)

print(f"Pipeline B clusters: {adata_nopca.obs['leiden'].nunique()}")

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_nopca, color=["dataset"], size=5, alpha=0.9)

In [ ]:
# Save / reload
adata_nopca.write('output/adata_plot_harmony_no_pca_2_brains.h5ad')
# adata_nopca = ad.read('output/adata_plot_harmony_no_pca_2_brains.h5ad')

---
# Part 2: Clustering Method Sensitivity

Apply Louvain, K-Means, Hierarchical (Ward), and Spectral on the PCA -> Harmony
embedding from Pipeline A. Cluster count is matched to Leiden's k.

In [ ]:
# Determine k from Leiden so K-Means / Hierarchical / Spectral get the same target.
n_leiden = adata_pca.obs['leiden'].nunique()
print(f"Number of Leiden clusters: {n_leiden} (used as k for K-Means / Hierarchical / Spectral)")

In [ ]:
# Louvain on the PCA Harmony graph (uses neighbors already in adata_pca).
sc.tl.louvain(adata_pca, resolution=1.1, random_state=38)
print(f"Louvain clusters: {adata_pca.obs['louvain'].nunique()}")

In [ ]:
# K-Means on the PCA Harmony embedding.
X_harmony_pca = adata_pca.obsm['X_harmony']

kmeans = KMeans(n_clusters=n_leiden, random_state=42, n_init=10)
adata_pca.obs['kmeans'] = kmeans.fit_predict(X_harmony_pca).astype(str)
print(f"K-Means clusters: {adata_pca.obs['kmeans'].nunique()}")

In [ ]:
# Hierarchical (Ward) -- subsample then assign remaining cells via KNN.
max_hier = 50000

print(f"Subsampling {max_hier} cells for Hierarchical clustering (full data: {adata_pca.n_obs})")
np.random.seed(42)
sub_idx = np.random.choice(adata_pca.n_obs, max_hier, replace=False)
X_sub = X_harmony_pca[sub_idx]

ward = AgglomerativeClustering(n_clusters=n_leiden, linkage='ward')
sub_labels = ward.fit_predict(X_sub)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_sub, sub_labels)
adata_pca.obs['hierarchical'] = knn.predict(X_harmony_pca).astype(str)
print(f"Hierarchical clusters: {adata_pca.obs['hierarchical'].nunique()}")

In [ ]:
# Spectral clustering -- subsample if too large, then assign remaining cells.
n_cells = adata_pca.n_obs
max_spectral = 50000

if n_cells > max_spectral:
    print(f"Subsampling {max_spectral} cells for Spectral clustering (full data: {n_cells})")
    np.random.seed(42)
    sub_idx = np.random.choice(n_cells, max_spectral, replace=False)
    X_sub = X_harmony_pca[sub_idx]

    spectral = SpectralClustering(n_clusters=n_leiden, random_state=42,
                                   affinity='nearest_neighbors', n_neighbors=10)
    sub_labels = spectral.fit_predict(X_sub)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_sub, sub_labels)
    adata_pca.obs['spectral'] = knn.predict(X_harmony_pca).astype(str)
else:
    spectral = SpectralClustering(n_clusters=n_leiden, random_state=42,
                                   affinity='nearest_neighbors', n_neighbors=10)
    adata_pca.obs['spectral'] = spectral.fit_predict(X_harmony_pca).astype(str)

print(f"Spectral clusters: {adata_pca.obs['spectral'].nunique()}")

---
# Part 3: Pairwise ARI and NMI across all methods

We align `adata_pca` (PCA pipeline + all extra methods) and `adata_nopca`
(no-PCA Leiden) by `obs_names` rather than positional indexing.  Both AnnData
objects were built from the same `adata_both` so they share the same cell
identities, but a row-order divergence at any earlier step (re-read from
disk, etc.) would silently break a positional comparison -- aligning by
obs_names is bug-proof.

In [ ]:
# Pairwise sensitivity: PCA-pipeline methods only (leiden_noPCA is treated
# separately as the dimensionality-reduction comparison further down).
# Both adata_pca and adata_nopca were built from the same adata_both, so
# obs_names align; we still align by name for safety.
common = adata_pca.obs_names.intersection(adata_nopca.obs_names)
print(f"Cells common to both pipelines: {len(common)} / {adata_pca.n_obs}")

# Reindex each label column to the common cell order. Reindex returns NaN if
# a cell is missing; we drop those rows from the comparison.
def _aligned(adata, key):
    s = pd.Series(adata.obs[key].astype(str).values,
                  index=adata.obs_names.astype(str))
    return s.reindex(common.astype(str))

labels = {
    'leiden':       _aligned(adata_pca, 'leiden'),
    'louvain':      _aligned(adata_pca, 'louvain'),
    'kmeans':       _aligned(adata_pca, 'kmeans'),
    'hierarchical': _aligned(adata_pca, 'hierarchical'),
    'spectral':     _aligned(adata_pca, 'spectral'),
}
labels_df = pd.DataFrame(labels)
valid_mask = labels_df.notna().all(axis=1).values
print(f"Cells with valid labels for every method: {valid_mask.sum()} / {len(labels_df)}")

label_vecs = {k: labels_df[k].values[valid_mask] for k in labels_df.columns}

In [ ]:
# Pairwise ARI / NMI across all clusterings.
method_names = list(label_vecs.keys())
n = len(method_names)

ari_matrix = np.ones((n, n))
nmi_matrix = np.ones((n, n))

for i in range(n):
    for j in range(i + 1, n):
        a = label_vecs[method_names[i]]
        b = label_vecs[method_names[j]]
        ari = adjusted_rand_score(a, b)
        nmi = normalized_mutual_info_score(a, b)
        ari_matrix[i, j] = ari_matrix[j, i] = ari
        nmi_matrix[i, j] = nmi_matrix[j, i] = nmi

ari_df = pd.DataFrame(ari_matrix, index=method_names, columns=method_names)
nmi_df = pd.DataFrame(nmi_matrix, index=method_names, columns=method_names)

print("Pairwise ARI:")
print(ari_df.round(3))
print("\nPairwise NMI:")
print(nmi_df.round(3))

In [ ]:
# ARI / NMI heatmap (matches Random_Clustering_reproducibility layout).
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(ari_df, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            square=True, ax=axes[0], cbar_kws={'shrink': 0.8},
            annot_kws={'size': 20})
axes[0].set_title('Adjusted Rand Index (ARI)', fontsize=20)
axes[0].tick_params(axis='both', labelsize=18)
axes[0].collections[0].colorbar.ax.tick_params(labelsize=18)

sns.heatmap(nmi_df, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.8},
            annot_kws={'size': 20})
axes[1].set_title('Normalized Mutual Information (NMI)', fontsize=20)
axes[1].tick_params(axis='both', labelsize=18)
axes[1].collections[0].colorbar.ax.tick_params(labelsize=18)

plt.suptitle('Clustering Sensitivity: 5 methods on the PCA + Harmony embedding',
             fontsize=20, y=1.02)
plt.tight_layout()
plt.savefig('clustering_sensitivity_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('clustering_sensitivity_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics (off-diagonal pairs of the 5-method PCA-pipeline matrix).
ari_vals = ari_matrix[np.triu_indices(n, k=1)]
nmi_vals = nmi_matrix[np.triu_indices(n, k=1)]

print(f"All pairwise comparisons ({len(ari_vals)} pairs, PCA pipeline only):")
print(f"  ARI: mean={ari_vals.mean():.4f}, min={ari_vals.min():.4f}, "
      f"max={ari_vals.max():.4f}, std={ari_vals.std():.4f}")
print(f"  NMI: mean={nmi_vals.mean():.4f}, min={nmi_vals.min():.4f}, "
      f"max={nmi_vals.max():.4f}, std={nmi_vals.std():.4f}")

# Dimensionality-reduction comparison (PCA Leiden vs no-PCA Leiden) is a
# separate one-off computation, not part of the pairwise matrix above.
leiden_pca   = _aligned(adata_pca,   'leiden').values
leiden_nopca = _aligned(adata_nopca, 'leiden').values
mask_dimred  = pd.notna(leiden_pca) & pd.notna(leiden_nopca)
ari_dimred = adjusted_rand_score(leiden_pca[mask_dimred], leiden_nopca[mask_dimred])
nmi_dimred = normalized_mutual_info_score(leiden_pca[mask_dimred], leiden_nopca[mask_dimred])
print(f"\nPCA Leiden vs no-PCA Leiden (dimensionality-reduction sensitivity):")
print(f"  ARI = {ari_dimred:.4f}")
print(f"  NMI = {nmi_dimred:.4f}")

ari_df.to_csv('sensitivity_ARI_matrix.csv')
nmi_df.to_csv('sensitivity_NMI_matrix.csv')
print("\nSaved sensitivity_ARI_matrix.csv, sensitivity_NMI_matrix.csv")

---
# Part 4: 1st-brain RNA marker characterization

Build a 1st-brain RNA AnnData, attach the cluster labels from every method,
run Wilcoxon DE and collect top markers per cluster.  All downstream
section-5 export and section-6 composite-figure cells reuse this object.

In [ ]:
# === Section-4 configuration (mirrors Random_Clustering_reproducibility) ===
OUTPUT_DIR = 'output/'
OUTPUT_PREFIX = ''            # e.g. 'dimred_' to isolate outputs

def _outpath(name):
    return os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}{name}")

# Marker-gene computation
N_TOP_MARKERS = 5
MARKER_MIN_CELLS = 10

# Composite-figure inputs
COMPOSITE_EXAMPLE_KEYS = ['leiden', 'louvain', 'kmeans',
                          'hierarchical', 'spectral', 'leiden_noPCA']
COMPOSITE_MIN_CELLS = 500
SPATIAL_ROTATION_DEG = -137

# Heatmap clustering
LINKAGE_METHOD = 'ward'
DISTANCE_METRIC = 'euclidean'
N_ROW_GROUPS = 3
N_COL_GROUPS = 4
MIN_GROUP_SIZE = 2

# Styling
DOT_SIZE = 0.4
TITLE_FS = 32
AXIS_LABEL_FS = 26
TICK_FS = 26
CLUSTER_LABEL_FS = 16
HEATMAP_TICK_FS = 16
CBAR_LABEL_FS = 26

if LINKAGE_METHOD == 'ward' and DISTANCE_METRIC != 'euclidean':
    print(f"[note] LINKAGE_METHOD='ward' requires DISTANCE_METRIC='euclidean'; overriding.")
    DISTANCE_METRIC = 'euclidean'

print(f"Output directory: {os.path.abspath(OUTPUT_DIR)!r}")
print(f"Output prefix:    {OUTPUT_PREFIX!r}")

In [ ]:
# Build the RNA AnnData and attach every clustering label.
# The analysis brain (matched to rna_df) is dataset='2nd_brain'; the
# Harmony / Leiden labels for those cells live in adata_pca / adata_nopca.
import warnings
warnings.filterwarnings("ignore")

ANALYSIS_BRAIN_KEY = '2nd_brain'   # the brain that has matched RNA data

# Mask + obs_names of analysis-brain cells in each pipeline.
mask_pca   = (adata_pca.obs['dataset']   == ANALYSIS_BRAIN_KEY).values
mask_nopca = (adata_nopca.obs['dataset'] == ANALYSIS_BRAIN_KEY).values
obs_names_pca   = adata_pca.obs_names[mask_pca].astype(str)
obs_names_nopca = adata_nopca.obs_names[mask_nopca].astype(str)

# Cells common to all three sources (protein PCA, protein no-PCA, RNA).
common_cells = (
    pd.Index(obs_names_pca)
      .intersection(pd.Index(obs_names_nopca))
      .intersection(pd.Index(rna_df.index))
)
if len(common_cells) == 0:
    raise RuntimeError(
        f"No overlap between {ANALYSIS_BRAIN_KEY} cells (PCA), "
        f"{ANALYSIS_BRAIN_KEY} cells (no PCA), and rna_df. "
        f"Check that the obs_names prefix on rna_df matches the prefix used in the concat step."
    )
rna_df_sub = rna_df.loc[common_cells]
print(f"{ANALYSIS_BRAIN_KEY} cells used for RNA characterization: {len(common_cells)}")

# Build the RNA AnnData and normalize.
adata_rna = ad.AnnData(
    X=rna_df_sub.to_numpy(),
    obs=pd.DataFrame(index=common_cells),
    var=pd.DataFrame(index=rna_df_sub.columns.astype(str)),
)
sc.pp.normalize_total(adata_rna, inplace=True)
sc.pp.log1p(adata_rna)

# Position arrays for slicing obsm of adata_pca / adata_nopca.
pca_positions   = adata_pca.obs_names.astype(str).get_indexer(common_cells)
nopca_positions = adata_nopca.obs_names.astype(str).get_indexer(common_cells)

# Spatial coords from adata_pca (same cells, same spatial), UMAP from PCA pipeline.
adata_rna.obsm['spatial'] = adata_pca.obsm['spatial'][pca_positions]
adata_rna.obsm['X_umap']  = adata_pca.obsm['X_umap'][pca_positions]

# Cluster labels: 5 methods on PCA Harmony + no-PCA Leiden.
# Drop any cells whose label is missing in either pipeline (NaN -> string 'nan'
# after astype(str), so explicitly filter those out before building categoricals).
for key in ['leiden', 'louvain', 'kmeans', 'hierarchical', 'spectral']:
    lab = np.asarray(adata_pca.obs[key].astype(str).values)[pca_positions]
    lab[lab == 'nan'] = np.nan
    adata_rna.obs[key] = pd.Categorical(lab)

lab_nopca = np.asarray(adata_nopca.obs['leiden'].astype(str).values)[nopca_positions]
lab_nopca[lab_nopca == 'nan'] = np.nan
adata_rna.obs['leiden_noPCA'] = pd.Categorical(lab_nopca)

label_keys = ['leiden', 'louvain', 'kmeans', 'hierarchical', 'spectral', 'leiden_noPCA']

# Run rank_genes_groups for each label set, collect top markers per cluster.
top_markers = {}
for key in label_keys:
    counts = adata_rna.obs[key].value_counts()
    keep = counts[counts >= MARKER_MIN_CELLS].index.tolist()
    sub = adata_rna[adata_rna.obs[key].isin(keep)].copy()
    sub.obs[key] = sub.obs[key].cat.remove_unused_categories()
    sc.tl.rank_genes_groups(sub, groupby=key, method='wilcoxon', key_added=f'rank_{key}')
    res = sub.uns[f'rank_{key}']
    groups = res['names'].dtype.names
    top_markers[key] = {
        g: [str(res['names'][g][i]).strip("'\"") for i in range(N_TOP_MARKERS)]
        for g in groups
    }
    adata_rna.uns[f'rank_{key}'] = res

print(f"adata_rna: {adata_rna.n_obs} cells x {adata_rna.n_vars} genes")
print(f"Label sets: {label_keys}")
for k in label_keys:
    print(f"  {k}: {len(top_markers[k])} clusters with top markers")

---
# Part 5: Export RNA z-score matrices per clustering (for R heatmap pipeline)

For each clustering result write the same pair of CSVs that `heatmap_RNA.R`
expects (matching `Step_8_1`):

* `R_expression_zscore_{key}.csv` -- clusters x marker genes, z-scored per gene
* `R_marker_labels_{key}.csv`     -- per-cluster top marker + cell count

In [ ]:
# Export RNA z-score matrix + marker labels per clustering.
var_clean = [str(v).strip("'\"") for v in adata_rna.var_names]
var_to_col = {v: i for i, v in enumerate(var_clean)}

for key in label_keys:
    counts = adata_rna.obs[key].value_counts()
    keep = counts[counts >= MARKER_MIN_CELLS].index.tolist()
    sub = adata_rna[adata_rna.obs[key].isin(keep)].copy()
    sub.obs[key] = sub.obs[key].cat.remove_unused_categories()
    clusters = sub.obs[key].cat.categories.tolist()

    # Marker pool: union of top markers across all clusters in this set.
    marker_pool = []
    seen = set()
    for cl in clusters:
        for g in top_markers[key].get(cl, []):
            if g not in seen and g in var_to_col:
                marker_pool.append(g)
                seen.add(g)
    if not marker_pool:
        print(f"  {key}: no marker genes available, skipping")
        continue

    gene_idx = [var_to_col[g] for g in marker_pool]

    # Mean expression per cluster, restricted to marker_pool columns.
    expr_rows = []
    for cl in clusters:
        cells = sub[sub.obs[key] == cl]
        X = cells.X.toarray() if hasattr(cells.X, 'toarray') else cells.X
        expr_rows.append(np.asarray(X[:, gene_idx]).mean(axis=0).flatten())
    expr_df = pd.DataFrame(expr_rows, index=clusters, columns=marker_pool)

    # Z-score per gene (column-wise) -- matches Step_8_1 convention.
    expr_z = expr_df.apply(zscore, axis=0)

    z_path = _outpath(f'R_expression_zscore_{key}.csv')
    expr_z.to_csv(z_path)

    labels_df_out = pd.DataFrame({
        'marker': [top_markers[key].get(cl, ['?'])[0] for cl in clusters],
        'cell_count': [int((sub.obs[key] == cl).sum()) for cl in clusters],
    }, index=clusters)
    l_path = _outpath(f'R_marker_labels_{key}.csv')
    labels_df_out.to_csv(l_path)

    print(f"  {key}: clusters={len(clusters)}, markers={len(marker_pool)} -> {z_path}, {l_path}")

---
# Part 6: 3-column composite figure (UMAP / spatial / RNA z-score heatmap)

One row per clustering method.  Cluster colors are synchronized between the
UMAP (col 1) and spatial (col 2) panels within each row; the heatmap (col 3)
shows mean per-cluster expression of the union of top-5 markers across all
methods, z-scored per gene and reordered by hierarchical clustering.

In [ ]:
# 3-column composite figure (matches Random_Clustering_reproducibility.ipynb).
# Wrapped in a function so we can render two versions: (1) the four non-Leiden
# clustering methods, (2) PCA Leiden vs no-PCA Leiden, saved as separate files.

# Per-method palette seeds. Different seeds produce visually distinct tab20
# permutations even when two methods share the same cluster count, so louvain
# and the no-PCA Leiden don't look identical to the PCA Leiden row.
PALETTE_SEEDS = {
    'leiden':       0,
    'louvain':      7,
    'kmeans':       0,
    'hierarchical': 0,
    'spectral':     0,
    'leiden_noPCA': 13,
}


def _merge_small_groups(grp_in_order, min_size=MIN_GROUP_SIZE):
    grp = np.asarray(grp_in_order).copy()
    if min_size <= 1:
        return grp
    for _ in range(grp.size):
        u, c = np.unique(grp, return_counts=True)
        small = set(u[c < min_size].tolist())
        if not small:
            break
        snap = grp.copy()
        for i in range(len(grp)):
            if snap[i] not in small:
                continue
            left  = snap[i - 1] if i > 0 else None
            right = snap[i + 1] if i < len(grp) - 1 else None
            candidates = [g for g in (left, right) if g is not None and g != snap[i]]
            if not candidates:
                continue
            best = max(candidates, key=lambda g: int(np.sum(snap == g)))
            grp[i] = best
    return grp


def _shuffled_palette(n, seed=0):
    base = plt.cm.get_cmap("tab20", max(n, 20))
    rng = np.random.default_rng(seed)
    perm = rng.permutation(max(n, 20))
    return [base(int(p)) for p in perm[:n]]


# Rotate spatial coords once (used by every composite render).
_theta = np.radians(SPATIAL_ROTATION_DEG)
_R = np.array([[np.cos(_theta), -np.sin(_theta)],
               [np.sin(_theta),  np.cos(_theta)]])
adata_rna.obsm['spatial_rotated'] = adata_rna.obsm['spatial'] @ _R.T


def render_composite(keys, out_basename):
    """Render the 3-column composite (UMAP / spatial / RNA z-score heatmap)
    for a given ordered list of clustering keys and save it to disk."""
    n_rows = len(keys)

    # Shared marker pool across the rows in this render so the heatmap
    # x-axis is comparable between panels.
    shared_pool, seen = [], set()
    for key in keys:
        for cl, genes in top_markers.get(key, {}).items():
            for g in genes:
                if g not in seen:
                    shared_pool.append(g); seen.add(g)
    shared_pool = [g for g in shared_pool if g in var_to_col]
    gene_idx = [var_to_col[g] for g in shared_pool]

    fig, axes = plt.subplots(
        n_rows, 3,
        figsize=(30, 8 * n_rows),
        gridspec_kw={'width_ratios': [1, 1.4, 2.5]},
    )
    if n_rows == 1:
        axes = axes.reshape(1, 3)

    for r, key in enumerate(keys):
        labels_cat = adata_rna.obs[key].astype('category')
        cats_all = labels_cat.cat.categories
        counts = labels_cat.value_counts()
        keep = [c for c in cats_all if int(counts.get(c, 0)) >= COMPOSITE_MIN_CELLS]
        cats = pd.Index(keep)
        n_clust = len(cats)

        palette = _shuffled_palette(n_clust, seed=PALETTE_SEEDS.get(key, 0))
        cluster_colors = {c: palette[i] for i, c in enumerate(cats)}
        keep_mask_all = labels_cat.isin(cats).values

        # Col 1 -- UMAP
        ax = axes[r, 0]
        umap_xy = adata_rna.obsm['X_umap']
        for cat in cats:
            m = (labels_cat == cat).values
            ax.scatter(umap_xy[m, 0], umap_xy[m, 1], s=DOT_SIZE,
                       c=[cluster_colors[cat]], alpha=0.4, rasterized=True)
            cx, cy = umap_xy[m, 0].mean(), umap_xy[m, 1].mean()
            ax.text(cx, cy, str(cat), fontsize=CLUSTER_LABEL_FS, fontweight='bold',
                    ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.7, lw=0))
        ax.set_title(f'{key} -- UMAP ({n_clust} clusters)',
                      fontsize=TITLE_FS)
        ax.set_xlabel('UMAP1', fontsize=AXIS_LABEL_FS)
        ax.set_ylabel('UMAP2', fontsize=AXIS_LABEL_FS)
        ax.tick_params(axis='both', labelsize=TICK_FS)

        # Col 2 -- Spatial (rotated)
        ax = axes[r, 1]
        xy = adata_rna.obsm['spatial_rotated']
        for cat in cats:
            m = (labels_cat == cat).values
            ax.scatter(xy[m, 0], xy[m, 1], s=DOT_SIZE,
                       c=[cluster_colors[cat]], alpha=0.5, rasterized=True)
        ax.set_title(f'{key} -- Spatial',
                      fontsize=TITLE_FS)
        ax.set_xlabel('X', fontsize=AXIS_LABEL_FS)
        ax.set_ylabel('Y', fontsize=AXIS_LABEL_FS)
        ax.tick_params(axis='both', labelsize=TICK_FS)
        ax.set_aspect('equal'); ax.invert_yaxis()

        # Col 3 -- RNA z-score heatmap with hierarchical-clustering reorder.
        sub = adata_rna[keep_mask_all].copy()
        sub.obs[key] = sub.obs[key].astype('category').cat.remove_unused_categories()
        clusters = sub.obs[key].cat.categories.tolist()

        expr_rows = []
        for cl in clusters:
            cells = sub[sub.obs[key] == cl]
            X = cells.X.toarray() if hasattr(cells.X, 'toarray') else cells.X
            expr_rows.append(np.asarray(X[:, gene_idx]).mean(axis=0).flatten())
        expr_df = pd.DataFrame(expr_rows, index=clusters, columns=shared_pool)
        z_df = expr_df.apply(zscore, axis=0).fillna(0.0)

        row_link = linkage(pdist(z_df.values,   metric=DISTANCE_METRIC), method=LINKAGE_METHOD)
        col_link = linkage(pdist(z_df.values.T, metric=DISTANCE_METRIC), method=LINKAGE_METHOD)
        row_order = leaves_list(row_link)
        col_order = leaves_list(col_link)
        z_ord = z_df.iloc[row_order, :].iloc[:, col_order]

        # Cut the dendrograms into groups and locate break positions in the
        # reordered axes so we can draw pheatmap-style separator lines.
        row_grp = fcluster(row_link, t=N_ROW_GROUPS, criterion='maxclust')[row_order]
        col_grp = fcluster(col_link, t=N_COL_GROUPS, criterion='maxclust')[col_order]
        row_grp = _merge_small_groups(row_grp, MIN_GROUP_SIZE)
        col_grp = _merge_small_groups(col_grp, MIN_GROUP_SIZE)
        row_breaks = np.where(np.diff(row_grp) != 0)[0] + 1
        col_breaks = np.where(np.diff(col_grp) != 0)[0] + 1

        ax = axes[r, 2]
        sns.heatmap(
            z_ord, ax=ax, cmap='Blues',
            vmin=float(z_ord.values.min()), vmax=float(z_ord.values.max()),
            cbar_kws={'shrink': 0.5, 'label': 'z-score'},
            xticklabels=True, yticklabels=True,
            linewidths=0,                  # no per-cell gridlines
        )
        ax.grid(False)                     # suppress any inherited matplotlib grid
        for b in row_breaks:
            ax.axhline(b, color='white', linewidth=2)
        for b in col_breaks:
            ax.axvline(b, color='white', linewidth=2)
        ax.set_title(
            f'{key} -- RNA z-score',
            fontsize=TITLE_FS,
        )
        ax.tick_params(axis='x', labelsize=HEATMAP_TICK_FS, rotation=90)
        ax.tick_params(axis='y', labelsize=HEATMAP_TICK_FS, rotation=0)
        cbar = ax.collections[0].colorbar
        if cbar is not None:
            cbar.ax.tick_params(labelsize=TICK_FS)
            cbar.set_label('z-score', fontsize=CBAR_LABEL_FS)

    plt.tight_layout()
    plt.savefig(_outpath(f'{out_basename}.pdf'), dpi=200, bbox_inches='tight')
    plt.savefig(_outpath(f'{out_basename}.png'), dpi=200, bbox_inches='tight')
    plt.show()


# Clustering-method sensitivity figure: the four non-Leiden methods.
# (Leiden + no-PCA Leiden go into a separate figure below.)
render_composite(
    ['louvain', 'kmeans', 'hierarchical', 'spectral'],
    'clustering_3col_composite',
)

# Dimensionality-reduction sensitivity figure: PCA Leiden vs no-PCA Leiden.
render_composite(
    ['leiden', 'leiden_noPCA'],
    'clustering_3col_composite_leiden_pca_vs_nopca',
)